# Simulation de Monte Carlo

## 1- Contextualisation et Principe de Base

La **Simulation de Monte Carlo (MC)** est une méthode statistique puissante utilisée pour modéliser des systèmes complexes ou des processus où le hasard joue un rôle important. En finance, elle est indispensable pour évaluer le **risque** et valoriser des instruments dont la valeur dépend de la trajectoire future incertaine d'un ou plusieurs actifs sous-jacents.

Le principe est de **générer des milliers (ou millions) de scénarios aléatoires** de l'évolution des prix futurs, puis de calculer la moyenne des résultats obtenus. Plus le nombre de scénarios ($N$) est grand, plus l'estimation se rapproche de la vraie valeur théorique.


#### Rappel : Valorisation d'une Option Européenne

Une **Option Européenne** est un produit dérivé simple qui donne à son détenteur le droit d’acheter (**Call**) ou de vendre (**Put**) un actif sous-jacent (comme une action) à un prix fixé (**Strike**, noté $K$) à une date unique, la **Maturité** ($T$).

Le prix d'aujourd'hui ($V_0$) est donné par l'**espérance mathématique actualisée** du payoff futur de l'option, dans un monde neutre au risque :

$$
V_0 = e^{-rT} \mathbb{E}[\text{Payoff}(S_T)]
$$

* $V_0$ : Prix de l'option à $t=0$.
* $e^{-rT}$ : Facteur d'actualisation continue au taux sans risque ($r$).
* $\mathbb{E}[\cdot]$ : Espérance (Moyenne des scénarios simulés).
* $\text{Payoff}(S_T)$ : Valeur de l'option à l'échéance $T$.

## 2 - Le Modèle Stochastique du Prix : Mouvement Brownien Géométrique (GBM)

Pour valoriser l'option par Monte Carlo, nous devons modéliser comment le prix de l'actif sous-jacent ($S_t$) évolue dans le temps. Le modèle standard en finance pour le prix des actions est le **Mouvement Brownien Géométrique (GBM)** .

Le GBM suppose que les rendements de l'actif sont distribués selon une loi normale et que l'évolution du prix est continue et aléatoire.

### Formule Discrète pour la Simulation Monte Carlo

En finance computationnelle, on utilise la solution analytique du GBM pour simuler directement le prix final ($S_T$) à partir du prix initial ($S_0$) :

$$
S_T = S_0 \cdot \exp \left( \left( r - \frac{1}{2}\sigma^2 \right) T + \sigma \sqrt{T} Z \right)
$$

* $S_0$ : **Prix initial de l'actif** (Stock Price).
* $T$ : **Temps jusqu'à maturité** (en années).
* $r$ : **Taux sans risque** (Risk-free rate).
* $\sigma$ : **Volatilité** (mesure de l'incertitude).
* $Z$ : Une variable aléatoire tirée d'une **distribution normale standard** ($\mathcal{N}(0, 1)$), que l'on génère par la fonction `np.random.standard_normal()`.

L'expression $\left( r - \frac{1}{2}\sigma^2 \right) T$ est la composante **déterministe** (le *drift*).
L'expression $\sigma \sqrt{T} Z$ est la composante **stochastique** (le choc aléatoire).

## 3 - Étapes Détaillées de la Simulation pour la Valorisation

La simulation Monte Carlo pour une option européenne à l'échéance $T$ se décompose en trois étapes de calcul cruciales.

### 3.1. Génération des Trajectoires des Prix (`simulate_price_paths`)

Cette fonction génère un vecteur de $N$ prix finaux possibles ($S_T$) en utilisant la formule discrète du GBM (ci-dessus) et en injectant $N$ valeurs aléatoires distinctes pour $Z$.

### 3.2. Calcul des Payoffs à Maturité (`calculate_payoffs`)

À la date d'échéance $T$, la valeur de l'option (le **payoff**) est déterminée par son prix final $S_T$ par rapport au prix d'exercice $K$.

Les payoffs sont définis comme suit :

| Type d'Option | Formule du Payoff | Signification |
| :--- | :--- | :--- |
| **Call Européen** | $\max(S_T - K, 0)$ | Le gain est le maximum entre la différence de prix et zéro. Le porteur gagne si $S_T > K$. |
| **Put Européen** | $\max(K - S_T, 0)$ | Le gain est le maximum entre la différence et zéro. Le porteur gagne si $S_T < K$. |

### 3.3. Actualisation et Calcul du Prix Final (`discount_payoffs` et `calculate_option_price`)

#### Actualisation
Puisque les payoffs sont perçus dans le futur ($T$), ils doivent être ramenés à leur valeur actuelle ($t=0$) en utilisant le taux sans risque $r$.

$$\text{Payoff Actualisé} = \text{Payoff} \times e^{-rT}$$

#### Calcul du Prix
Le prix de l'option est la moyenne de tous les payoffs actualisés simulés. C'est l'estimation de l'espérance mathématique :

$$
V_0 \approx \frac{1}{N} \sum_{i=1}^{N} (\text{Payoff Actualisé})_i
$$

Où $N$ est le nombre de simulations (**`n_simulations`**).

In [24]:
# Utilisation de la librairie Numpy 
import numpy as np 

#### Etape 1 : Définition des paramètres
Définir une fonction pour déclarer toutes les variables

In [26]:
def define_parameters():
    option_type = "call" # Type d'option 'call' ou 'put'
    K = 10.0             # Strike 
    S0 = 8.0             # Prix du sous-jacent en 0 
    r = 0.0325           # Taux sans risque 
    vol = 0.2            # volatilité du sous-jacent (généralement une vol à 1 an)
    T = 2.0              # Durée en année de détention de l'option 
    n = 1000000          # nombre de variable aléatoire (nombre de simulation de monte carlo 
    return option_type, S0, r, K, vol, T, n 

#### Etape 2 : Simulation des scénarios de prix futurs
Définir une fonction pour simuler les prix futurs de l'actif sous-jacent à l'échéance T en utilisant une distribution normale pour le facteur de bruit.

In [28]:
def simulate_price_paths(S0, r, vol, T, N):
    nrand = np.random.normal(0, 1, N)  # Génération de variables normales standards 
    ST = S0 * np.exp((r - 0.5 * vol**2) * T + vol * np.sqrt(T) * nrand)
    return ST

#### Etape 3 : Calcul des payoffs pour chaque scénario
Créer une fonction pour calculer le payoff de l'option call ou d'un put pour chaque scénario simulé

In [30]:
def calculate_payoffs(ST, K, option_type):
    if option_type == 'call' : 
        payoff = np.maximum(ST - K,0) 
    else :
        payoff = np.maximum(K - ST,0) 

    return payoff 

#### Etape 4 : Actualisation des payoffs
Créer une fonction qui actualise les payoffs simulés au taux sans risque

In [32]:
def discount_payoffs(payoffs, r, T):
    payoff_act = np.exp(-r*T) * payoffs
    return payoff_act 

#### Etape 5 : Calcul de la moyenne des payoffs actualisés 
Créer une fonction pour calculer le prix de l'option comme la moyenne des payoffs actualisés

In [34]:
def calculate_option_price(discounted_payoffs):
    OP = np.mean(discounted_payoffs)
    return OP 

#### Etape 6 : Coder le script qui appelle toutes ces fonctions 

In [36]:
# Programme principal
if __name__ == "__main__":
    # Ajout des paramètres 
    option_type, S0, r, K, vol, T, n =  define_parameters()
    
    # Simuler les prix 
    ST = simulate_price_paths(S0, r, vol, T, n) 
    print(f"Liste des prix futures : {ST}")
    
    # Calculer les payoffs
    Payoffs = calculate_payoffs(ST, K, option_type) 
    print(f"Liste des payoffs : {Payoffs}")

    # Actualisation des payoffs 
    payoffs_actu = discount_payoffs(Payoffs, r, T)
    print(f"Liste des payoffs actualisés : {payoffs_actu}")

    # Calcul final 
    Prix = calculate_option_price(payoffs_actu)
    print(f"Prix de l'option : {Prix}")

Liste des prix futures : [12.28286367 11.41674356 13.66275984 ...  7.06706373 12.59298498
  5.43327255]
Liste des payoffs : [2.28286367 1.41674356 3.66275984 ... 0.         2.59298498 0.        ]
Liste des payoffs actualisés : [2.13919727 1.32758429 3.43225307 ... 0.         2.42980186 0.        ]
Prix de l'option : 0.4398123013073707


Fin....